In [1]:
import os
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
import pandas as pd

In [2]:
load_dotenv()
entsoe_api_key = os.getenv("ENTSOE_API_KEY")

client = EntsoePandasClient(api_key=entsoe_api_key)

start = pd.Timestamp("20260801", tz="Europe/Warsaw")
end = pd.Timestamp("20260901", tz="Europe/Warsaw")
country_code = "PL"

# ENTSO-E Transparency Platform

The ENTSO-E Transparency Platform is the central electricity market transparency platform operated by the European Network of Transmission System Operators for Electricity (ENTSO-E) in accordance with Commission Regulation (EU) No. 543/2013. It provides standardized electricity-system and wholesale-market data collected from Transmission System Operators (TSOs), power exchanges, and other qualified data providers across Europe.

## Description of Data from the ENTSO-E API

The following table provides an overview of the main variables obtained from the ENTSO-E API. The selection and terminology are based primarily on the official [Manual of Procedures (MoP)](https://www.entsoe.eu/data/transparency-platform/mop/#mop-revisions) and the corresponding data publications of the ENTSO-E Transparency Platform.

The **day-ahead electricity price** is the **target variable** in the model. All other variables are used as model features.

| Category | Data | API Method Name | Unit | Market Time Unit (MTU) |
|---|---|---|---|---|
| **Information relating to the use of cross-zonal capacities** | Day-ahead electricity prices **(dependent variable)** | `query_day_ahead_prices` | EUR/MWh | 15 min / 1 hour |
| | Physical cross-border flow (export) | `query_physical_crossborder_allborders` (`export=True`) | MW | 15 min / 1 hour |
| | Physical cross-border flow (import) | `query_physical_crossborder_allborders` (`export=False`) | MW | 15 min / 1 hour |
| **Information on total load** | Total load per bidding zone per market time unit | `query_load` | MW | 15 min / 1 hour |
| | Day-ahead forecast of the total load per market time unit | `query_load_forecast` | MW | 15 min / 1 hour |
| **Forecast generation** | Day ahead aggregated generation | `query_generation_forecast` | MW | 15 min / 1 hour |
|| Day ahead generation forecast for wind and solar | `query_wind_and_solar_forecast` | MW | 15 min / 1 hour |
| **Actual generation** | Aggregated generation per generation type | `query_generation` | MW | 15 min / 1 hour |
|| Pumped storage/reservoir stored energy | `query_aggregate_water_reservoirs_and_hydro_storage` | MWh |  1 week |
| **Information relating to the unavailability of generation and production units** | Planned and actual unavailability of a generation unit | `query_unavailability_of_generation_units` | MW | 1 hour |
| | Planned and actual unavailability of a production unit | `query_unavailability_of_production_units` | MW | 1 hour |
| **Imbalance** | Imbalance prices | `query_imbalance_prices` | currency/MWh | 15 min / 1 hour |
|| Total imbalance volume | `query_imbalance_volumes` | MW | 15 min / 1 hour |

**Note:** The Single Day-Ahead Coupling (SDAC) transitioned from hourly to 15-minute Market Time Units (MTUs) in September 2025. The go-live took place on trading day **30 September 2025**, with the first delivery day under the new 15-minute MTU being **1 October 2025**. For the purposes of this project, all data available at a 15-minute resolution will be aggregated to a **1-hour resolution**. This ensures a consistent resolution across all variables used in the model.

## 1. Information relating to the use of cross zonal capacities

### 1.1 Energy Prices

**Description:**

Day-ahead electricity prices for each bidding zone, expressed in €/MWh at the market time-unit resolution. These prices represent the outcome of the day-ahead market and are the target variable for the forecasting analysis.

**Publication deadline for ENTSO-E:**

It shall be published no later than one hour after gate closure.

In [3]:
day_ahead_prices = client.query_day_ahead_prices(
    country_code=country_code, start=start, end=end
)
pd.DataFrame(day_ahead_prices, columns=["day_ahead_prices"])

,day_ahead_prices
2026-08-01 00:00:00+02:00,184.07
2026-08-01 00:15:00+02:00,173.93
2026-08-01 00:30:00+02:00,164.92
2026-08-01 00:45:00+02:00,156.69
2026-08-01 01:00:00+02:00,170.00
...,...
2026-08-31 23:00:00+02:00,154.85
2026-08-31 23:15:00+02:00,162.69
2026-08-31 23:30:00+02:00,151.99
2026-08-31 23:45:00+02:00,139.43


### 1.2 Physical Flows (cross-border)

**Description:**

Physical flows between bidding zones per market time unit.

**Publication deadline for Entso-E:**

At the latest H+1 after the end of the operating period

In [4]:
physical_crossborder_allborders_export = client.query_physical_crossborder_allborders(
    country_code=country_code, start=start, end=end, export=True
)
physical_crossborder_allborders_export

,CZ,DE_LU,LT,SE_4,SK,UA,sum
2026-08-01 00:00:00+02:00,868.63,0.000,0.0,0.0,347.1,0.0,1215.730
2026-08-01 00:15:00+02:00,857.57,0.000,0.0,0.0,366.6,0.0,1224.170
2026-08-01 00:30:00+02:00,907.95,0.000,0.0,0.0,435.4,0.0,1343.350
2026-08-01 00:45:00+02:00,924.76,0.000,0.0,0.0,455.3,0.0,1380.060
2026-08-01 01:00:00+02:00,1084.69,0.000,0.0,0.0,508.1,0.0,1592.790
...,...,...,...,...,...,...,...
2026-08-31 22:45:00+02:00,430.32,0.000,0.0,0.0,764.0,230.5,1424.820
2026-08-31 23:00:00+02:00,429.99,0.000,0.0,0.0,786.5,240.3,1456.790
2026-08-31 23:15:00+02:00,444.12,32.260,0.0,0.0,842.0,225.4,1543.780
2026-08-31 23:30:00+02:00,477.84,141.630,0.0,0.0,889.9,262.4,1771.770


In [5]:
physical_crossborder_allborders_import = client.query_physical_crossborder_allborders(
    country_code=country_code, start=start, end=end, export=False
)
physical_crossborder_allborders_import

,CZ,DE_LU,LT,SE_4,SK,UA,sum
2026-08-01 00:00:00+02:00,0.0,660.245,208.1,487.9,0.0,61.7,1417.945
2026-08-01 00:15:00+02:00,0.0,705.092,200.3,487.8,0.0,40.8,1433.992
2026-08-01 00:30:00+02:00,0.0,739.642,187.7,487.7,0.0,12.7,1427.742
2026-08-01 00:45:00+02:00,0.0,698.588,216.1,487.6,0.0,32.7,1434.988
2026-08-01 01:00:00+02:00,0.0,536.591,181.2,487.7,0.0,22.5,1227.991
...,...,...,...,...,...,...,...
2026-08-31 22:45:00+02:00,0.0,934.002,179.8,199.0,0.0,0.0,1312.802
2026-08-31 23:00:00+02:00,0.0,907.560,78.3,199.4,0.0,0.0,1185.260
2026-08-31 23:15:00+02:00,0.0,787.839,106.0,199.1,0.0,0.0,1092.939
2026-08-31 23:30:00+02:00,0.0,816.800,9.7,198.8,0.0,0.0,1025.300


### 1.3 Scheduled exchanges from explicit and implicit allocations

## 2. Information on total load

### 2.1 Total load per bidding zone per market time unit
**Description:** 

Actual total electricity load for each bidding zone, expressed in MW at the market time-unit resolution. This variable represents the realized electricity demand and can be used as an explanatory variable in the forecasting analysis.

**Publication deadline for ENTSO-E:**

Publication based on market time unit. At the latest H+1 after the end of the operating period (of one market time unit length).

In [6]:
load = client.query_load(country_code=country_code, start=start, end=end)
load

,Actual Load
2026-08-01 00:00:00+02:00,16166.005
2026-08-01 00:15:00+02:00,16054.105
2026-08-01 00:30:00+02:00,15575.064
2026-08-01 00:45:00+02:00,15369.113
2026-08-01 01:00:00+02:00,15136.911
...,...
2026-08-31 22:45:00+02:00,16926.615
2026-08-31 23:00:00+02:00,16759.399
2026-08-31 23:15:00+02:00,16698.687
2026-08-31 23:30:00+02:00,16088.294


### 2.2 Day-ahead forecast of the total load per market time unit

**Description:**

Day-ahead forecasts of total electricity load for each bidding zone, expressed in MW at the market time-unit resolution. These forecasts are published before delivery and therefore represent information that can be available at the forecast time.

**Publication deadline for ENTSO-E:**

Publication is necessary in due time for the negotiation of all transactions: D-1, at the latest 2 hours before the gate closure time of the day-ahead market in the bidding area. If the gate closure doesn’t exists in the bidding area then the publication time is D-1, at 12:00 in local time zone.

In [7]:
load_forecast = client.query_load_forecast(
    country_code=country_code, start=start, end=end
)
load_forecast

,Forecasted Load
2026-08-01 00:00:00+02:00,15750.0
2026-08-01 00:15:00+02:00,15450.0
2026-08-01 00:30:00+02:00,15150.0
2026-08-01 00:45:00+02:00,14950.0
2026-08-01 01:00:00+02:00,14750.0
...,...
2026-08-31 22:45:00+02:00,17250.0
2026-08-31 23:00:00+02:00,16950.0
2026-08-31 23:15:00+02:00,16650.0
2026-08-31 23:30:00+02:00,16300.0


## 3. Forecast generation

### 3.1 Day ahead aggregated generation

**Detailed description:**

An estimate of the total scheduled generation (MW) per bidding zone, per each market time unit of the following day.

**Publication deadline for ENTSO-E**

D-1 at 18h00 the latest in Brussels time.

In [8]:
agg_generation_forecast = client.query_generation_forecast(
    country_code=country_code, start=start, end=end
)
agg_generation_forecast

2026-08-01 00:00:00+02:00    15274.0
2026-08-01 00:15:00+02:00    15186.0
2026-08-01 00:30:00+02:00    15238.0
2026-08-01 00:45:00+02:00    15151.0
2026-08-01 01:00:00+02:00    15351.0
                              ...   
2026-08-31 22:45:00+02:00    17258.0
2026-08-31 23:00:00+02:00    17082.0
2026-08-31 23:15:00+02:00    17051.0
2026-08-31 23:30:00+02:00    16962.0
2026-08-31 23:45:00+02:00    16597.0
Freq: 15min, Name: Actual Aggregated, Length: 2976, dtype: float64

### 3.2 Day ahead generation forecasts for wind and solar

**Detailed description:**

A forecast of wind and solar power generation (MW) per bidding zone, per each market time unit of the following day.

**Publication deadline for ENTSO-E**

D-1 not later than 18h00 in Brussels time

In [9]:
wind_solar_forecast = client.query_wind_and_solar_forecast(
    country_code=country_code, start=start, end=end
)
wind_solar_forecast

,Solar,Wind Offshore,Wind Onshore
2026-08-01 00:00:00+02:00,0.0,1.9700,1580.7320
2026-08-01 00:15:00+02:00,0.0,1.4880,1566.8205
2026-08-01 00:30:00+02:00,0.0,1.0320,1579.1755
2026-08-01 00:45:00+02:00,0.0,0.6015,1578.2440
2026-08-01 01:00:00+02:00,0.0,0.3535,1579.2620
...,...,...,...
2026-08-31 22:45:00+02:00,0.0,102.4825,2652.1265
2026-08-31 23:00:00+02:00,0.0,111.4540,2729.6550
2026-08-31 23:15:00+02:00,0.0,123.4850,2783.1175
2026-08-31 23:30:00+02:00,0.0,135.4315,2871.2550


## 4. Actual generation

### 4.1 Aggregated generation per type

**Detailed description:**

Actual electricity generation aggregated by generation type for each bidding zone, expressed in MW at the market time-unit resolution. The data describe realized generation from different technologies, including renewable and conventional sources.

**Publication deadline for ENTSO-E:**

H+1 following the concerned MTU

In [10]:
generation = client.query_generation(country_code=country_code, start=start, end=end)
generation

,Hydro Pumped Storage,Biomass,Fossil Brown coal/Lignite,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and pondage,Hydro Water Reservoir,Other renewable,Solar,Wind Offshore,Wind Onshore,Other
,Actual Consumption,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated,Actual Aggregated
2026-08-01 00:00:00+02:00,156.469,317.171,4267.124,174.020,2040.740,6614.833,141.964,133.6,53.888,18.410,55.691,0.0,0.0,1810.161,580.062
2026-08-01 00:15:00+02:00,160.113,316.533,4306.205,174.205,2090.508,6447.649,142.530,0.0,53.688,18.410,55.190,0.0,0.0,1680.859,575.683
2026-08-01 00:30:00+02:00,160.461,314.318,4303.181,173.461,2110.577,6190.106,143.256,0.0,53.688,18.410,55.070,0.0,0.0,1741.899,573.712
2026-08-01 00:45:00+02:00,160.171,317.115,4264.094,173.362,2246.898,6283.130,142.848,0.0,53.789,18.410,55.172,0.0,0.0,1467.349,574.400
2026-08-01 01:00:00+02:00,160.447,315.860,4313.158,173.708,2298.186,6367.026,143.853,0.0,53.665,18.235,54.865,0.0,0.0,1355.746,571.802
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-31 22:45:00+02:00,4.398,376.149,3746.284,126.061,2313.605,6025.597,122.098,715.1,61.254,20.306,56.196,0.0,21.2,2906.797,562.708
2026-08-31 23:00:00+02:00,4.360,371.797,3877.471,123.529,2345.111,6105.303,121.668,471.9,60.434,20.173,55.689,0.0,81.0,2934.630,555.877
2026-08-31 23:15:00+02:00,4.360,373.164,3792.018,125.270,2197.577,6176.907,120.892,468.3,60.434,100.173,55.915,0.0,82.0,2959.993,558.521


### 4.2 Pumped storage/reservoir stored energy

**Detailed description:**

Aggregated weekly average filling rate of all water reservoir and hydro storage plants (MWh) per bidding zone including the figure for the same week of the previous year.

**Publication deadline for ENTSO-E:**

End of the third working day of W+1.

In [11]:
water_hydro = client.query_aggregate_water_reservoirs_and_hydro_storage(
    country_code="AT", start=start, end=end
)
water_hydro

2026-07-26 22:00:00+00:00    1508323.53
2026-08-02 22:00:00+00:00    1583685.63
2026-08-09 22:00:00+00:00    1676650.10
2026-08-16 22:00:00+00:00    1691902.94
2026-08-23 22:00:00+00:00    1603915.99
2026-08-30 22:00:00+00:00    1776742.51
Freq: 7D, dtype: float64

## 5. Information relating to the unavailability of generation and production units

### 5.1 Planned and Current Unavailability of Generation Units

**Description:**

Planned unavailability of 100 MW or more of a generation unit, including changes of 100 MW or more in the planned unavailability of that generation unit, expected to last for at least one market time unit and up to three years ahead. The dataset also covers changes of 100 MW or more in the actual availability of a generation unit, expected to last for at least one market time unit.

**Publication Deadline for ENTSO-E:**

For planned unavailability, the information shall be published H+1 at the latest after the plan is approved. For changes in actual availability, the information shall be published no later than H+1 after the change in actual availability.

In [12]:
unavailability_of_generation_units = client.query_unavailability_of_generation_units(
    country_code=country_code, start=start, end=end
)
unavailability_of_generation_units

,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
created_doc_time,,,,,,,,,,,,,,,,,,
2025-10-05 16:31:05+02:00,0,PL,Planned maintenance,A03,None,2026-08-31 12:00:00+02:00,2kCxt9iEXPmDpEVqvDSL9w,91.4,Fossil Hard coal,19W000000000107C,Polska,EC Łódź-4,Łódź-4 B03,1,MAW,PT1M,1,2026-08-04 07:01:00+02:00
2025-10-06 13:38:03+02:00,0,PL,Planned maintenance,A03,Cancelled,2026-08-20 00:00:00+02:00,mQmes-XWN9f-UhdUepktZQ,226.0,Fossil Hard coal,19W0000000001519,Polska,Połaniec,Połaniec B4,1,MAW,PT1M,4,2026-07-05 00:01:00+02:00
2025-10-07 01:28:14+02:00,0,PL,Planned maintenance,A03,None,2026-08-28 00:00:00+02:00,TCJod7mQ5DATJrP0yAUsOg,105.9,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,Siekierki B08,1,MAW,PT1M,2,2026-03-02 00:01:00+01:00
2025-10-07 14:49:00+02:00,467.419,PL,Planned maintenance,A03,Cancelled,2026-09-01 00:00:00+02:00,H1GEK9DxJMTM3hvkoIDyng,620.0,Fossil Gas,19W000000000283T,Polska,Płock,Płock B01,1,MAW,PT1M,2,2026-08-01 00:00:00+02:00
2025-10-07 23:22:29+02:00,0,PL,Planned maintenance,A03,Cancelled,2026-09-08 00:00:00+02:00,GtHSAl6EK-QwQ9HF1xhhqA,209.0,Fossil Hard coal,19W0000000001713,Polska,Rybnik,Rybnik B5,1,MAW,PT1M,5,2026-06-30 00:00:00+02:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-09-02 08:54:38+02:00,0,PL,Unplanned outage,A03,None,2026-09-02 08:45:00+02:00,fI2_HXIOPdhvVyUZF08AMg,224.0,Fossil Hard coal,19W0000000001519,Polska,Połaniec,Połaniec B5,1,MAW,PT1M,7,2026-08-07 23:50:00+02:00
2026-09-02 17:36:22+02:00,0,PL,Unplanned outage,A03,None,2026-09-04 00:00:00+02:00,CRYDsj49UjZghHTD5zjkeA,205.9,Fossil Hard coal,19W0000000000636,Polska,Jaworzno 3,Jaworzno 3 B5,1,MAW,PT1M,3,2026-08-31 19:02:00+02:00
2026-09-04 13:28:00+02:00,0,PL,Unplanned outage,A03,None,2026-09-11 00:00:00+02:00,lyRhqYHF00hGCTiZ8my92Q,139.1,Fossil Hard coal,19W0000000002361,Polska,Siersza,Siersza B2,1,MAW,PT1M,2,2026-08-31 22:37:00+02:00


### 5.2 Planned and Current Unavailability of Production Units

**Description:**

Planned unavailability of a production unit with an installed generation capacity of 200 MW or more, including changes of 100 MW or more in the planned unavailability of that production unit, but not already published as a generation-unit unavailability. The event is expected to last for at least one market time unit and up to three years ahead. The dataset also covers changes of 100 MW or more in the actual availability of a production unit with an installed generation capacity of 200 MW or more.

**Publication Deadline for ENTSO-E:**

For planned unavailability, the information shall be published H+1 at the latest after the plan is approved. For changes in actual availability, the information shall be published no later than H+1 after the change in actual availability.

In [13]:
unavailability_of_production_units = client.query_unavailability_of_production_units(
    country_code=country_code, start=start, end=end
)
unavailability_of_production_units

,avail_qty,biddingzone_domain,businesstype,curvetype,docstatus,end,mrid,nominal_power,plant_type,production_resource_id,production_resource_location,production_resource_name,production_resource_psr_name,pstn,qty_uom,resolution,revision,start
created_doc_time,,,,,,,,,,,,,,,,,,
2025-10-07 01:28:18+02:00,441,PL,Planned maintenance,A03,Cancelled,2026-08-15 00:00:00+02:00,Bazhc8LjP9Z-ZOWDLRAy1g,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,3,2026-06-12 00:01:00+02:00
2025-10-07 01:28:18+02:00,473,PL,Planned maintenance,A03,Cancelled,2026-08-15 00:00:00+02:00,A4bzEQ9CvIvYcgtxTRan-Q,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-07-18 00:01:00+02:00
2025-10-07 01:28:18+02:00,505,PL,Planned maintenance,A03,Cancelled,2026-08-18 00:00:00+02:00,os347y4910SslQTN4bsoqQ,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-08-15 00:01:00+02:00
2025-10-07 01:28:19+02:00,473,PL,Planned maintenance,A03,Cancelled,2026-08-18 00:00:00+02:00,dNyXiN4oHkpre1YvVzuwUA,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,3,2026-06-12 00:01:00+02:00
2025-10-07 01:28:19+02:00,505,PL,Planned maintenance,A03,None,2026-08-18 00:00:00+02:00,5az3Te-5EIACvlWo89amXg,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,1,2026-06-29 00:01:00+02:00
2025-10-07 01:28:19+02:00,493,PL,Planned maintenance,A03,Cancelled,2026-08-25 00:00:00+02:00,hbBoiUjwyimXCt8_zefgOw,533.8,Fossil Hard coal,19W0000000002191,Polska,EC Siekierki,,1,MAW,PT1M,2,2026-08-24 00:01:00+02:00
2025-10-07 01:28:26+02:00,105,PL,Planned maintenance,A03,Cancelled,2026-08-22 00:00:00+02:00,8Kk-abmiUHINsDfOK20QDQ,245.8,Fossil Hard coal,19W000000000297I,Polska,EC Żerań 1,,1,MAW,PT1M,2,2026-07-20 00:01:00+02:00


## 6. Imbalance

### 6.1 Imbalance prices

**Description:**

Imbalance prices per balancing time unit.

**Publication deadline for ENTSO-E:**

As soon as possible and no later than 30 minutes after the end of the ISP.

In [14]:
imbalance_volumes = client.query_imbalance_volumes(
    country_code=country_code,
    start=start,
    end=end,
)

imbalance_volumes

2026-08-01 00:00:00+02:00    -90.847
2026-08-01 00:15:00+02:00    -45.470
2026-08-01 00:30:00+02:00    -62.256
2026-08-01 00:45:00+02:00    -23.274
2026-08-01 01:00:00+02:00    -96.201
                              ...   
2026-08-31 22:45:00+02:00    -86.939
2026-08-31 23:00:00+02:00   -135.107
2026-08-31 23:15:00+02:00   -185.728
2026-08-31 23:30:00+02:00   -124.854
2026-08-31 23:45:00+02:00   -158.604
Name: Imbalance Volume, Length: 2976, dtype: float64

### 6.2 Total imbalance volume

**Description:**

Total imbalance volume per balancing time unit.

**Publication deadline for ENTSO-E:**

As soon as possible and no later than 30 minutes after the end of the ISP.

In [15]:
imbalance_prices = client.query_imbalance_prices(
    country_code=country_code,
    start=start,
    end=end,
)

imbalance_prices

,Long,Short
2026-08-01 00:00:00+02:00,894.70,894.70
2026-08-01 00:15:00+02:00,651.55,651.55
2026-08-01 00:30:00+02:00,731.77,731.77
2026-08-01 00:45:00+02:00,678.40,678.40
2026-08-01 01:00:00+02:00,769.58,769.58
...,...,...
2026-08-31 22:45:00+02:00,823.59,823.59
2026-08-31 23:00:00+02:00,860.26,860.26
2026-08-31 23:15:00+02:00,936.51,936.51
2026-08-31 23:30:00+02:00,761.54,761.54
